# BSL modeling — data pipeline

Turns the preprocessing manifest into padded, model-ready batches: loads
features from the **packed memmap** file, tokenizes the Bangla text with
**SentencePiece BPE**, splits the data, and exposes `SignDataset` /
`collate_fn` / DataLoaders.

Features are normalized + reduced to the upper-body landmarks at load time via
`bsl_utils.prepare_features` (so this notebook never touches MediaPipe).

## 1. Imports and configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm

from bsl_utils import prepare_features, feature_dim

ROOT = "/Volumes/Easha/SSD Backup/Bornil_Dataset/archive"
BASE = "/Volumes/Easha/SSD Backup/Bornil_Dataset"
MANIFEST_PATH = os.path.join(ROOT, "processed_manifest.csv")
PACKED_PATH = os.path.join(BASE, "features_packed_v2.npy")            # single memmap file
LENGTHS_PATH = os.path.join(BASE, "features_packed_v2_lengths.npy")
SP_MODEL = os.path.join(ROOT, "bsl_sp.model")                         # SentencePiece BPE

SEED = 42
BATCH_SIZE = 4

print("manifest       :", os.path.exists(MANIFEST_PATH))
print("packed features:", os.path.exists(PACKED_PATH))
print("sp model       :", os.path.exists(SP_MODEL))

## 2. Load manifest + packed features

In [ ]:
df = pd.read_csv(MANIFEST_PATH)
print("Rows:", len(df), "| Columns:", list(df.columns))

# Features live in ONE packed memmap file (17,988 small files -> 1, gentle on the
# NTFS driver). row.pack_idx indexes into it; the real per-clip frame count is in
# the lengths array. mmap_mode='r' pages data in on demand (low RAM).
FEATURES = np.load(PACKED_PATH, mmap_mode="r")     # (N, 128, 258) float32
LENGTHS = np.load(LENGTHS_PATH)                     # (N,) real frame counts
print("packed:", FEATURES.shape, FEATURES.dtype, "| lengths:", LENGTHS.shape)
assert len(df) == FEATURES.shape[0] == len(LENGTHS), "manifest / pack size mismatch"
df.head(3)

## 3. Subword tokenizer (SentencePiece BPE)

Word-level targets are unlearnable here — 25k unique words, 58% appearing once.
A ~4k BPE vocabulary gives frequent, reusable subword units instead. The model
(`bsl_sp.model`) is trained once from the manifest text and then reused.

In [ ]:
# Train the BPE model once (from the manifest text), then load it.
if not os.path.exists(SP_MODEL):
    _corpus = os.path.join(ROOT, "_sp_corpus.txt")
    with open(_corpus, "w") as f:
        for s in df["clean_text"]:
            f.write(str(s).strip() + "\n")
    spm.SentencePieceTrainer.train(
        input=_corpus, model_prefix=os.path.join(ROOT, "bsl_sp"),
        vocab_size=4000, model_type="bpe", character_coverage=1.0,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3, minloglevel=2,
    )
    os.remove(_corpus)

sp = spm.SentencePieceProcessor(model_file=SP_MODEL)
PAD_ID, BOS_ID, EOS_ID = sp.pad_id(), sp.bos_id(), sp.eos_id()
VOCAB_SIZE = sp.get_piece_size()


def encode_sentence(text):
    return [BOS_ID] + sp.encode(str(text)) + [EOS_ID]


def decode_tokens(token_ids):
    ids = [int(i) for i in token_ids if int(i) not in (PAD_ID, BOS_ID, EOS_ID)]
    return sp.decode(ids)


df["token_ids"] = df["clean_text"].apply(encode_sentence)

print("subword vocab size:", VOCAB_SIZE, "| pad/bos/eos =", PAD_ID, BOS_ID, EOS_ID)
print("Example :", df.iloc[0]["clean_text"])
print("Pieces  :", sp.encode(df.iloc[0]["clean_text"], out_type=str)[:12], "...")
print("Encoded :", df.iloc[0]["token_ids"][:12], "...")
print("Decoded :", decode_tokens(df.iloc[0]["token_ids"]))

## 4. Train / val / test split

In [ ]:
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(df))

n_val = max(1, int(0.1 * len(df)))
n_test = max(1, int(0.1 * len(df)))
val_idx = perm[:n_val]
test_idx = perm[n_val:n_val + n_test]
train_idx = perm[n_val + n_test:]

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)
print("train / val / test:", len(train_df), len(val_df), len(test_df))

## 5. Dataset, collate, and loaders

`SignDataset` slices a clip's frames straight from the packed memmap
(`FEATURES[pack_idx]`, trimmed to its real length) and applies
`prepare_features` (normalize + upper-body select) to get `(T, 218)`.
`collate_fn` pads variable-length clips and subword-token sequences per batch.

In [ ]:
class SignDataset(Dataset):
    def __init__(self, dataframe, normalize=True, upper_body=True):
        self.df = dataframe.reset_index(drop=True)
        self.normalize = normalize
        self.upper_body = upper_body

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        i = int(row["pack_idx"])
        x = np.array(FEATURES[i, :LENGTHS[i]])               # (T, 258) from packed memmap
        x = prepare_features(x, normalize=self.normalize, upper_body=self.upper_body)  # (T, 218)
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(row["token_ids"], dtype=torch.long)
        return x, y


def collate_fn(batch):
    xs, ys = zip(*batch)
    x_lengths = torch.tensor([x.shape[0] for x in xs], dtype=torch.long)
    y_lengths = torch.tensor([y.shape[0] for y in ys], dtype=torch.long)
    xs_padded = torch.nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=0.0)
    ys_padded = torch.nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=PAD_ID)
    return xs_padded, ys_padded, x_lengths, y_lengths


# Size the model from these, never a hard-coded literal.
INPUT_DIM = feature_dim(upper_body=True)   # 218
print("INPUT_DIM:", INPUT_DIM, "| VOCAB_SIZE:", VOCAB_SIZE)

train_loader = DataLoader(SignDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(SignDataset(val_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(SignDataset(test_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
x, y, x_lengths, y_lengths = next(iter(train_loader))
print("x:", tuple(x.shape), " # (batch, Tmax, INPUT_DIM)")
print("y:", tuple(y.shape), " # (batch, max_tokens)")
print("x_lengths:", x_lengths.tolist())
print("y_lengths:", y_lengths.tolist())

## Next step: the model

Everything the model needs is defined:
- `train_loader` / `val_loader` / `test_loader` — padded batches
  `(batch, Tmax, INPUT_DIM)` plus integer subword-token targets.
- `INPUT_DIM` (= 218) — per-frame feature width; size the encoder from this.
- `VOCAB_SIZE` (= 4000 BPE), `sp`, and `decode_tokens` — the target tokenizer.
- `x_lengths` / `y_lengths` and `PAD_ID` — for masking / CTC / packing.

Build a baseline here — e.g. a BiLSTM encoder over the keypoint sequence with a
CTC head (subword targets stay well under the 128 frame budget), or a small
encoder–decoder with attention.